In [0]:
from pyspark.sql import functions as F

In [1]:
dbutils.widgets.text("catalog", "")
dbutils.widgets.text("source_volume", "")

NameError: name 'dbutils' is not defined

In [ ]:
catalog = dbutils.widgets.get("catalog")
source_volume = dbutils.widgets.get("source_volume")

In [ ]:
if not catalog:
    raise ValueError("Required parameter 'catalog' was not provided")

if not source_volume:
    raise ValueError("Required parameter 'source_volume' was not provided")

In [ ]:
source_path = f"/Volumes/{catalog}/bronze/{source_volume}/"

bronze_table = f"{catalog}.bronze.customers"

checkpoint_path = (
    f"/Volumes/{catalog}/bronze/{source_volume}/_checkpoints/customers"
)

schema_location = (
    f"/Volumes/{catalog}/bronze/{source_volume}/_schemas/customers"
)

In [ ]:
print(f"Catalog       : {catalog}")
print(f"Source volume : {source_volume}")
print(f"Source path   : {source_path}")
print(f"Bronze table  : {bronze_table}")
print(f"Checkpoint    : {checkpoint_path}")


In [ ]:
# --------------------------------------------------
# 3. Read source files using Auto Loader
# --------------------------------------------------

customers_stream = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("cloudFiles.schemaLocation", schema_location)
        .option("cloudFiles.inferColumnTypes", "true")
        .load(source_path)
)


In [ ]:
bronze_customers = (
    customers_stream
        .withColumn("ingestion_timestamp", F.current_timestamp())
        .withColumn("source_file", F.col("_metadata.file_path"))
)

In [ ]:
query = (
    bronze_customers.writeStream
        .format("delta")
        .option("checkpointLocation", checkpoint_path)
        .trigger(availableNow=True)
        .toTable(bronze_table)
)

print("Customer Bronze ingestion completed.")